# Local notebook -> Modal function calling

## How to run

1. In a terminal, run `cd ~/Developer/ml-lab && uv sync --extra remote`.
2. Authenticate once with `uv run modal setup`.
3. Open this notebook locally in Cursor.
4. Choose the local `ML Lab (.venv)` Jupyter kernel.
5. Run the first two cells to load Modal and define the function.
6. Run the final cell to call the function on a remote Modal T4 GPU.

The notebook stays local; only the decorated function runs remotely. The final cell is billable. The remote and local Python versions are both 3.12.

In [ ]:
import modal

app = modal.App("ml-lab-notebook")
image = modal.Image.debian_slim(python_version="3.12").pip_install("torch")
print("Modal client loaded locally")

In [ ]:
@app.function(image=image, gpu="T4", timeout=300)
def remote_matmul(size: int = 1024) -> dict[str, object]:
    import torch

    left = torch.randn(size, size, device="cuda")
    right = torch.randn(size, size, device="cuda")
    result = left @ right
    return {
        "device": str(result.device),
        "gpu": torch.cuda.get_device_name(0),
        "shape": list(result.shape),
    }


print("Remote function defined locally")

In [ ]:
with modal.enable_output(), app.run():
    result = remote_matmul.remote(1024)

print(result)